## Setup

Keep [`example_data.py`](example_data.py) and [`missingness_fidelity_metrics.py`](missingness_fidelity_metrics.py) beside this notebook. The project environment supplies NumPy, pandas, Matplotlib, IPython, and scikit-learn.

In [ ]:
#| label: setup

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from example_data import make_dependence_example
from missingness_fidelity_metrics import (
    MIN_DEPENDENCE_CASES,
    missingness_dependence_details,
    missingness_dependence_gap,
)

## Learning goals

By the end of this lesson, you should be able to:

- explain what missingness–value dependence means;
- identify the target and predictors used for variable $j$;
- explain why both classifiers use the same $V$;
- calculate the AUROC gap for each eligible variable; and
- interpret positive, zero, and negative metric values.

## 1. Metric, inputs, and notation

The **Missingness–Value Dependence Gap** is a missingness-fidelity metric. We write it as $\mathrm{Gap}_{\mathrm{miss}}$. It checks whether missingness has a similar relationship with the other variables in $X$ and $Y$. It does not only compare how often values are missing.

We use the same notation as the previous lessons:

- $X$ is the reference dataset and $Y$ is the comparison dataset.
- $d$ is the number of variables shared by both datasets.
- $n_X$ and $n_Y$ are the numbers of records in $X$ and $Y$.
- $X_j=\{x_{ij}\}_{i=1}^{n_X}$ and $Y_j=\{y_{ij}\}_{i=1}^{n_Y}$ are variable $j$ in the two datasets.
- $M^X$ and $M^Y$ are the missingness indicator matrices of $X$ and $Y$.
- $V$ is the real missingness holdout, with dimensions $n_V\times d$.

Its missingness indicator matrix is $M^V$, with dimensions $n_V\times d$. The holdout is not used to train either classifier.

This metric uses the original incomplete datasets. It does not use $\widetilde X$, $\widetilde Y$, or $\widetilde V$. The model performs the required preprocessing internally.

In [ ]:
#| label: dependence-inputs

example = make_dependence_example()
X, Y, V = example.X, example.Y, example.V
M_X = X.isna().astype("int8")
M_Y = Y.isna().astype("int8")
M_V = V.isna().astype("int8")

sample_summary = pd.DataFrame(
    {
        "records": [len(X), len(Y), len(V)],
        "variables": [X.shape[1], Y.shape[1], V.shape[1]],
        "income_k missing rate": [
            M_X["income_k"].mean(),
            M_Y["income_k"].mean(),
            M_V["income_k"].mean(),
        ],
    },
    index=["X", "Y", "V"],
)
display(sample_summary.round(4))
print(f"d = {X.shape[1]}, n_X = {len(X)}, n_Y = {len(Y)}")
print(f"shape(M^X) = {M_X.shape}")
print(f"shape(M^Y) = {M_Y.shape}")
print(f"shape(M^V) = {M_V.shape}")

The small tables from the earlier lessons do not contain enough missing and observed cases for classifier evaluation. This lesson therefore uses a larger companion example. It has the same four variables and the same notation.

In $X$ and $V$, the chance that `income_k` is missing depends on `age` and `region`. In $Y$, its missing locations are generated independently of the other values. The three tables have similar overall missingness rates. The important difference is the relationship between missingness and the other variables.

## 2. Missingness as a prediction target

Consider variable $j$. Its missingness indicator is the prediction target. For record $i$ in $X$, the target is

$$
(M^X)_{ij}=
\begin{cases}
1, & x_{ij}\text{ is missing}\\
0, & x_{ij}\text{ is observed}
\end{cases}
$$

The predictors are all variables except variable $j$. We write them as $X_{-j}$. We exclude $X_j$ because its value directly reveals whether it is missing.

If the other variables can predict $(M^X)_{ij}$, missingness in variable $j$ depends on information elsewhere in the record. The same idea applies to $Y$.

In [ ]:
#| label: visible-missingness-dependence

age_group_rows = []
for dataset_name, frame in [("X", X), ("Y", Y), ("V", V)]:
    age_group = np.where(frame["age"] < 45, "age < 45", "age >= 45")
    grouped_rates = (
        pd.DataFrame(
            {
                "age group": age_group,
                "income_k missing": frame["income_k"].isna().astype(int),
            }
        )
        .groupby("age group")["income_k missing"]
        .mean()
    )
    for group, rate in grouped_rates.items():
        age_group_rows.append(
            {
                "dataset": dataset_name,
                "age group": group,
                "P(income_k missing)": rate,
            }
        )

age_group_summary = pd.DataFrame(age_group_rows).pivot(
    index="age group", columns="dataset", values="P(income_k missing)"
)
display(age_group_summary[["X", "Y", "V"]].round(4))

The table gives a simple view of the dependence. In $X$ and $V$, `income_k` is much more likely to be missing for older records. The difference is much smaller in $Y$. The classifier can use `age`, `region`, and `outcome` together, so it can detect more than this two-group summary.

## 3. Eligible variables and two classifiers

A variable needs enough examples of both target classes. In this implementation, variable $j$ is eligible when both $X$ and $V$ contain at least five missing cases and at least five observed cases for that variable. Let $\mathcal J$ be the set of eligible variables.

For every $j\in\mathcal J$, fit two classifiers:

$$
f_j^X:X_{-j}\longrightarrow(M^X)_{\cdot j}
$$

$$
f_j^Y:Y_{-j}\longrightarrow(M^Y)_{\cdot j}
$$

The first classifier learns from $X$. The second learns from $Y$. Both predict whether variable $j$ is missing from the other variables in the record.

The implementation uses a histogram gradient-boosting classifier. It fills missing numerical predictors with their median and adds their missingness indicators. It gives missing categorical predictors an explicit missing label and then one-hot encodes them. If $Y_j$ is always observed or always missing, the $Y$ side receives an AUROC of $0.5$.

In [ ]:
#| label: eligible-variables

eligibility_rows = []
for column in X.columns:
    x_missing = int(M_X[column].sum())
    v_missing = int(M_V[column].sum())
    x_observed = len(X) - x_missing
    v_observed = len(V) - v_missing
    eligibility_rows.append(
        {
            "variable": column,
            "X missing": x_missing,
            "X observed": x_observed,
            "V missing": v_missing,
            "V observed": v_observed,
            "eligible": min(
                x_missing, x_observed, v_missing, v_observed
            )
            >= MIN_DEPENDENCE_CASES,
        }
    )

eligibility = pd.DataFrame(eligibility_rows)
eligible_columns = eligibility.loc[eligibility["eligible"], "variable"].tolist()
display(eligibility)
print(f"Eligible set J = {eligible_columns}")

## 4. Evaluate both classifiers on the same holdout

Both classifiers receive the other variables from $V$. Their predictions are compared with the true holdout indicators $(M^V)_{\cdot j}$. Neither classifier is evaluated on its training records.

Let their holdout AUROCs be

$$
a_j^X=\operatorname{AUC}\left(f_j^X;V\right)
$$

$$
a_j^Y=\operatorname{AUC}\left(f_j^Y;V\right)
$$

AUROC measures ranking. A value of $0.5$ means chance-level ranking. A value near $1$ means strong prediction. A value below $0.5$ means the ranking tends to be reversed.

Using the same holdout makes $a_j^X$ and $a_j^Y$ directly comparable. Their per-variable gap is

$$
g_j=a_j^X-a_j^Y
$$

In [ ]:
#| label: calculate-dependence-gap

categorical_columns = ["region", "outcome"]
details = missingness_dependence_details(
    reference=X,
    comparison=Y,
    evaluation=V,
    categorical_columns=categorical_columns,
    random_state=42,
)
Gap_miss = missingness_dependence_gap(
    reference=X,
    comparison=Y,
    evaluation=V,
    categorical_columns=categorical_columns,
    random_state=42,
)

display(details.round(4))
print(f"Missingness–value dependence gap = {Gap_miss:.6f}")
assert details["column"].tolist() == eligible_columns
assert np.allclose(details["gap"], details["AUC_X"] - details["AUC_Y"])

In [ ]:
#| label: fig-dependence-auroc
#| fig-cap: "Holdout AUROC for classifiers trained on X and Y."

positions = np.arange(len(details))
width = 0.36
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(positions - width / 2, details["AUC_X"], width, label="trained on X")
ax.bar(positions + width / 2, details["AUC_Y"], width, label="trained on Y")
ax.axhline(0.5, color="black", linestyle="--", linewidth=1, label="chance")
ax.set(
    xlabel="Eligible variable",
    ylabel="AUROC on V",
    ylim=(0, 1),
    xticks=positions,
    xticklabels=details["column"],
)
ax.legend()
plt.show()

## 5. Average the per-variable gaps

The Missingness–Value Dependence Gap is the average of the eligible-variable gaps:

$$
\mathrm{Gap}_{\mathrm{miss}}(X,Y;V)=
\frac1{|\mathcal J|}\sum_{j\in\mathcal J}
\left(a_j^X-a_j^Y\right)
$$

The result lies between $-1$ and $1$. It is a signed gap, not a distance. It can be negative. If no variable is eligible, the metric is undefined and the implementation returns `NaN`.

In [ ]:
#| label: average-variable-gaps

Gap_miss_from_column_gaps = details["gap"].mean()
print(f"Mean of the per-variable gaps = {Gap_miss_from_column_gaps:.6f}")
assert np.isclose(Gap_miss, Gap_miss_from_column_gaps)
assert -1.0 <= Gap_miss <= 1.0

## 6. Interpretation

For this example, $\mathrm{Gap}_{\mathrm{miss}}(X,Y;V)\approx0.4654$. This is a large positive gap. The classifier trained on $X$ predicts the holdout missingness much better than the classifier trained on $Y$. Therefore, $Y$ has lost much of the relationship between `income_k` missingness and the other variables.

- A positive value means $X$ preserves the holdout dependence better than $Y$.
- A value near $0$ means the two sides have similar average predictive performance.
- A negative value means $Y$ predicts the holdout missingness better than $X$.
- A value near $1$ means a very large advantage for $X$.
- A value near $-1$ means a very large advantage for $Y$.
- The value is not the percentage of missing entries that differ.

A value near $0$ needs care. Both classifiers may be strong, or both may be weak. Positive and negative column gaps can also cancel. Always inspect the per-variable AUROCs and gaps.

In [ ]:
#| label: metric-sanity-checks

Gap_miss_same = missingness_dependence_gap(
    reference=X,
    comparison=X,
    evaluation=V,
    categorical_columns=categorical_columns,
    random_state=42,
)
Gap_miss_reversed = missingness_dependence_gap(
    reference=Y,
    comparison=X,
    evaluation=V,
    categorical_columns=categorical_columns,
    random_state=42,
)

print(f"Gap_miss(X, X; V) = {Gap_miss_same:.6f}")
print(f"Gap_miss(Y, X; V) = {Gap_miss_reversed:.6f}")
assert np.isclose(Gap_miss_same, 0.0)
assert np.isclose(Gap_miss_reversed, -Gap_miss)

## 7. Important limits

This metric is directional. $X$ is the reference dataset, and $V$ should represent the same real missingness process. Swapping $X$ and $Y$ changes their roles. When the eligible set stays the same, it also reverses the sign.

The result depends on the classifier and its settings. It also needs enough missing and observed cases. A weak model may miss a real dependence. A small holdout may give an unstable AUROC. This metric measures predictive dependence. It does not prove why values are missing.

## Takeaway

The Missingness–Value Dependence Gap follows five steps:

1. choose variables with enough missing and observed cases;
2. use each variable's missingness indicator as a target;
3. train one classifier on $X$ and one on $Y$;
4. evaluate both classifiers on the same $V$; and
5. average $\mathrm{AUROC}_X-\mathrm{AUROC}_Y$ across eligible variables.

A positive result means $X$ teaches the holdout missingness dependence better than $Y$. A negative result means $Y$ does better. A result near $0$ means their average predictive performance is similar.